# Kokoro voice blending — usage examples

Demonstrates `KokoroVoiceBlender` from `blend_voices.py`: loading voices, blending two or more of them via weighted SLERP, blending timbre/prosody independently, saving/reloading blends, and synthesizing test audio.

In [ ]:
from blend_voices import KokoroVoiceBlender

blender = KokoroVoiceBlender(out_dir="custom_voices", lang_code="i")
test_text = "Ciao, questa è una voce mescolata creata da più voci italiane."

## 1. Blend two voices

`blend()` takes a dict of `{voice_name: weight}` — weights don't need to sum to 1, they're normalized automatically.

In [ ]:
two_voice_blend = blender.blend({"hf_alpha": 0.5, "af_heart": 0.5})
two_voice_blend.shape

## 2. Blend three or more voices

Same `blend()` call, just add more entries to the dict. Voices are folded in one at a time via sequential weighted SLERP, so this works for any number of voices.

In [ ]:
three_voice_blend = blender.blend({
    "if_sara": 0.5,
    "im_nicola": 0.3,
    "hf_alpha": 0.2,
})
three_voice_blend.shape

## 3. Blend timbre and prosody independently, across any number of voices

`per_half_blend()` splits each 256-dim voice vector into timbre (first 128 dims, decoder) and prosody (last 128 dims, predictor), and blends each half with its own voice mix. The two mixes don't need to share voices — e.g. keep the timbre close to one speaker while pulling rhythm/intonation from several others.

In [ ]:
per_half = blender.per_half_blend(
    tim_weights={"if_sara": 0.8, "im_nicola": 0.2},   # timbre mostly Sara
    pro_weights={"im_nicola": 0.6, "hf_alpha": 0.4},  # prosody mixed Nicola + hf_alpha
)
per_half.shape

## 4. Build several named blends at once

`create_blends()` takes a dict of `name -> weights` and returns a dict of `name -> tensor`, checking shapes along the way. This is the batch version of `blend()` above.

In [ ]:
specs = {
    "sara50_nicola30_hf20": {"if_sara": 0.5, "im_nicola": 0.3, "hf_alpha": 0.2},
    "sara_nicola_5050": {"if_sara": 0.5, "im_nicola": 0.5},
}
blends = blender.create_blends(specs)

## 5. Save and reload blends

Blends are saved as `.pt` files under `out_dir` (default `custom_voices/`), so they can be reused later without recomputing.

In [ ]:
blender.save_all(blends)

# later, in this or another notebook/session:
reloaded = blender.load_saved("sara50_nicola30_hf20")
all_saved = blender.load_all_saved()
list(all_saved.keys())

## 6. Synthesize test audio

`synthesize()` registers one blend with the Kokoro pipeline and writes a `.wav`; `synthesize_all()` does this for a whole dict of blends. The pipeline is created lazily on first use and reused after that.

In [ ]:
# single blend
blender.synthesize("sara_nicola_5050", blends["sara_nicola_5050"], test_text)

# every blend in the dict
blender.synthesize_all(blends, test_text)